In [87]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/soil/plsda/soil.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1.32':'13.1']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.32':'13.1'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.32':'13.1'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-02-08 16:39:49,646 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-02-08 16:39:49,655 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

In [88]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

## Definição das Zonas Espectrais

In [89]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.19),
('S', 2.19, 2.55),
('Rh L + Ar', 2.55, 3.21),
('K', 3.21, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.37),
('Ti ka', 4.37, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.13),
('Fe ka', 6.13, 6.80),
('Fe kb', 6.80, 7.30),
('background1', 7.30, 7.91),
('Cu', 7.91, 8.20),
('background2', 8.20, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background3', 11.14, 12.55),
('sum Fe' , 12.55, 13.1)
]

## VIP, Regression Coefficients e SHAP (como no original)

In [90]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [91]:
shap_global_importance = pd.read_csv('shap_soil.csv', sep=';') # loading previously saved shap_unique_df

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,3.70,0.027751,Ca ka
1,6.42,0.013531,Fe ka
2,5.90,0.012230,Mn
3,1.74,0.010560,Si
4,4.46,0.006195,Ti ka
5,7.02,0.000310,Fe kb
6,5.02,0.000083,Ti kb
7,1.46,0.000082,Al
8,3.32,0.000021,K
9,2.02,0.000021,P


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [92]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zona 'Al': VE = 79.34%, dim = 15 variáveis
Zona 'Si': VE = 91.83%, dim = 12 variáveis
Zona 'P': VE = 62.02%, dim = 17 variáveis
Zona 'S': VE = 36.84%, dim = 18 variáveis
Zona 'Rh L + Ar': VE = 42.17%, dim = 33 variáveis
Zona 'K': VE = 83.48%, dim = 16 variáveis
Zona 'Ca ka': VE = 99.02%, dim = 16 variáveis
Zona 'Ca kb': VE = 74.05%, dim = 27 variáveis
Zona 'Ti ka': VE = 89.48%, dim = 19 variáveis
Zona 'Ti kb': VE = 82.21%, dim = 19 variáveis
Zona 'Cr': VE = 53.41%, dim = 33 variáveis
Zona 'Mn': VE = 91.12%, dim = 18 variáveis
Zona 'Fe ka': VE = 56.98%, dim = 34 variáveis
Zona 'Fe kb': VE = 55.98%, dim = 26 variáveis
Zona 'background1': VE = 32.70%, dim = 31 variáveis
Zona 'Cu': VE = 53.86%, dim = 15 variáveis
Zona 'background2': VE = 26.43%, dim = 125 variáveis
Zona 'Fe ka + Ti ka': VE = 34.74%, dim = 23 variáveis
Zona 'background3': VE = 18.96%, dim = 71 variáveis
Zona 'sum Fe': VE = 55.82%, dim = 28 variáveis

Scores DataFrame shape: (148, 20)


In [93]:
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred_cont
)

In [94]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 123 | Descartados: 37
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 122 | Descartados: 38
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 121 | Descartados: 39
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 121 | Descartados: 39
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 121 | 

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Ca ka > -1.73,Ca ka > -0.36,Ca ka > -1.73,Ca ka > -0.36
1,Ca ka > -0.36,Ca ka > -1.73,Ca ka > -1.02,Ca ka > -1.73
2,Ca ka > -1.02,Ca ka > -1.02,Ca ka > -0.36,Fe ka > -0.43
3,Fe ka > 0.86,Fe ka > 0.86,Fe ka > 0.86,Ca ka > -1.02
4,Fe ka > -0.43,Fe ka > -0.43,Ca kb > -0.37,Fe ka > -2.66
...,...,...,...,...
96,NaN,NaN,Cr <= -0.12,background2 > 0.12
97,NaN,NaN,Rh L + Ar <= 0.23,K <= 0.02
98,NaN,NaN,background3 <= -0.04,Ca kb <= -0.04
99,NaN,NaN,Class_A,Class_A


In [95]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -1.73,11.435302,Ca ka,-1.73,>
1,Fe ka > 0.86,4.800227,Fe ka,0.86,>
2,Ti ka > 0.93,2.650134,Ti ka,0.93,>
3,Mn > -0.65,2.592024,Mn,-0.65,>
4,Ca kb > -0.37,2.578133,Ca kb,-0.37,>
5,Si > -0.66,1.584730,Si,-0.66,>
6,Fe kb > 0.40,1.580886,Fe kb,0.40,>
7,K > -0.38,1.247210,K,-0.38,>
8,Al <= 0.37,1.233161,Al,0.37,<=
9,Ti kb > 0.26,1.186732,Ti kb,0.26,>


In [96]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zona 'Al': VE = 89.82%, dim = 15 variáveis
Zona 'Si': VE = 95.22%, dim = 12 variáveis
Zona 'P': VE = 61.00%, dim = 17 variáveis
Zona 'S': VE = 42.01%, dim = 18 variáveis
Zona 'Rh L + Ar': VE = 51.54%, dim = 33 variáveis
Zona 'K': VE = 88.55%, dim = 16 variáveis
Zona 'Ca ka': VE = 99.40%, dim = 16 variáveis
Zona 'Ca kb': VE = 72.71%, dim = 27 variáveis
Zona 'Ti ka': VE = 93.59%, dim = 19 variáveis
Zona 'Ti kb': VE = 88.32%, dim = 19 variáveis
Zona 'Cr': VE = 56.23%, dim = 33 variáveis
Zona 'Mn': VE = 93.26%, dim = 18 variáveis
Zona 'Fe ka': VE = 58.12%, dim = 34 variáveis
Zona 'Fe kb': VE = 55.52%, dim = 26 variáveis
Zona 'background1': VE = 34.82%, dim = 31 variáveis
Zona 'Cu': VE = 55.93%, dim = 15 variáveis
Zona 'background2': VE = 26.61%, dim = 125 variáveis
Zona 'Fe ka + Ti ka': VE = 39.81%, dim = 23 variáveis
Zona 'background3': VE = 19.45%, dim = 71 variáveis
Zona 'sum Fe': VE = 69.07%, dim = 28 variáveis


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ca ka > -1.73,11.435302,Ca ka,-1.73,>,-2.806273,140.0,0.001954,Ca ka > -2.806273
1,Ca ka > -0.36,11.146522,Ca ka,-0.36,>,-0.641296,49.0,0.004141,Ca ka > -0.641296
2,Ca ka > -1.02,8.941993,Ca ka,-1.02,>,-1.635267,143.0,0.000757,Ca ka > -1.635267
3,Fe ka > 0.86,4.800227,Fe ka,0.86,>,55.595005,114.0,0.001174,Fe ka > 55.595005
4,Fe ka > -0.43,4.167810,Fe ka,-0.43,>,-54.402608,14.0,0.002481,Fe ka > -54.402608
...,...,...,...,...,...,...,...,...,...
100,K <= 0.02,0.359152,K,0.02,<=,0.001685,19.0,0.000477,K <= 0.001685
101,background3 <= 0.04,0.347558,background3,0.04,<=,0.036170,57.0,0.000169,background3 <= 0.036170
102,background3 <= -0.04,0.000197,background3,-0.04,<=,-0.027035,120.0,0.000368,background3 <= -0.027035
103,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [97]:
import plotly.graph_objects as go

n = 62
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'background2':
  - Dimensão: 125 variáveis espectrais
  - Range de energias: 8.2 - 10.68
  - Variância explicada pela PC1: 26.61%


In [98]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='regression',
        metric='mean_abs_diff', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 123 | Descartados: 37
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 122 | Descartados: 38
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 121 | Descartados: 39
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 120 | Descartados: 40
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 121 | Descartados: 39
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): regression
Modo de perturbação: median
Fonte das estatísticas: full
Métrica: mean_abs_diff
Total de fo

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ca ka > -0.36,Ca ka > -0.36,Ca ka > -0.36,Ca ka > -0.36
1,Ca ka > -1.02,Mn > -0.15,Ca ka > -1.02,Mn > 0.64
2,Ca ka > -1.73,Ca ka > -1.02,Mn > -0.15,Ca ka > -1.02
3,Mn > -0.15,Ca ka > -1.73,Si > 0.13,Ca ka > -1.73
4,Mn > -0.38,Si > 0.13,Ca ka > -1.73,Mn > -0.15
...,...,...,...,...
124,Fe ka + Ti ka > -0.04,NaN,Fe ka + Ti ka > -0.15,Fe ka + Ti ka > -0.15
125,background1 <= 0.16,NaN,Class_A,Class_A
126,Fe ka + Ti ka > -0.15,NaN,Class_B,Class_B
127,Class_A,NaN,NaN,NaN


In [99]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ca ka > -0.36,0.233097
1,Ca ka > -1.02,0.169248
2,Ca ka > -1.73,0.140496
3,Mn > -0.15,0.123699
4,Si > 0.13,0.088038
...,...,...
118,background1 <= 0.05,0.001256
119,background1 > -0.03,0.001249
120,Fe ka + Ti ka > -0.04,0.001159
121,background1 > -0.16,0.001141


In [100]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -0.36,9.738977,Ca ka,-0.36,>
1,Mn > 0.64,8.149500,Mn,0.64,>
2,Si > 0.13,3.827714,Si,0.13,>
3,Fe ka <= -2.66,2.928575,Fe ka,-2.66,<=
4,Ti ka > 0.93,0.730139,Ti ka,0.93,>
5,Fe kb <= -0.07,0.398422,Fe kb,-0.07,<=
6,K > 0.02,0.367059,K,0.02,>
7,Al <= -0.09,0.347074,Al,-0.09,<=
8,Ca kb > -0.04,0.309303,Ca kb,-0.04,>
9,background2 <= -0.29,0.243803,background2,-0.29,<=


In [101]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ca ka > -0.36,9.738977,Ca ka,-0.36,>,-0.641296,49.0,0.004141,Ca ka > -0.641296
1,Mn > 0.64,8.149500,Mn,0.64,>,1.293607,104.0,0.014336,Mn > 1.293607
2,Ca ka > -1.02,6.401255,Ca ka,-1.02,>,-1.635267,143.0,0.000757,Ca ka > -1.635267
3,Ca ka > -1.73,5.019360,Ca ka,-1.73,>,-2.806273,140.0,0.001954,Ca ka > -2.806273
4,Mn > -0.15,4.895011,Mn,-0.15,>,-0.357226,56.0,0.003096,Mn > -0.357226
...,...,...,...,...,...,...,...,...,...
131,Fe ka + Ti ka > -0.04,0.007072,Fe ka + Ti ka,-0.04,>,-0.040286,48.0,0.000022,Fe ka + Ti ka > -0.040286
132,background1 > -0.16,0.006099,background1,-0.16,>,-0.141474,2.0,0.000536,background1 > -0.141474
133,Fe ka + Ti ka > -0.15,0.003055,Fe ka + Ti ka,-0.15,>,-0.137035,129.0,0.007042,Fe ka + Ti ka > -0.137035
134,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


In [102]:
import plotly.graph_objects as go

n = 1
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Mn':
  - Dimensão: 18 variáveis espectrais
  - Variância explicada pela PC1: 93.26%


In [103]:
# Permutation importance baseado em mudança nas predições do modelo PLS
# Medimos a média da diferença absoluta entre as predições originais e as predições
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Predições base do modelo PLS
baseline_pred = pls_model.predict(Xcalclass_prep)
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_pred = pls_model.predict(X_perm)
        diffs.append(np.mean(np.abs(baseline_pred - perm_pred)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance': importance_list
})
permutation_df.sort_values(by='Permutation_importance', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance', ascending=False)
permutation_unique_df

,energy,Permutation_importance,Zone
0,3.7,0.033093,Ca ka
1,1.74,0.020366,Si
2,6.42,0.019398,Fe ka
3,5.9,0.018689,Mn
4,4.46,0.018559,Ti ka
5,7.02,0.005123,Fe kb
6,1.48,0.003036,Al
7,5.02,0.002826,Ti kb
8,3.3,0.002801,K
9,4,0.002580,Ca kb


In [104]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'VIP_Score': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_Coefficient' : pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,VIP_Score,Reg_Coefficient,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Ca ka,Mn,Ca ka,Ca ka,Ca ka,Ca ka
1,Fe ka,Si,Fe ka,Si,Mn,Fe ka
2,Mn,Ca ka,Mn,Fe ka,Si,Ti ka
3,Si,Ti ka,Si,Mn,Fe ka,Mn
4,Fe kb,Fe ka,Ti ka,Ti ka,Ti ka,Ca kb
5,Ca kb,P,Fe kb,Fe kb,Fe kb,Si
6,Ti ka,K,Ti kb,Al,K,Fe kb
7,Al,Al,Al,Ti kb,Al,K
8,K,Ca kb,K,K,Ca kb,Al
9,sum Fe,sum Fe,P,Ca kb,background2,Ti kb


In [105]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_22248\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
1,VIP_Score,Shap,0.936470
11,Shap,LRC_covariance,0.859867
4,VIP_Score,LRC_covariance,0.855513
9,Shap,Permutation,0.811500
12,Permutation,LRC_perturbation,0.806490
10,Shap,LRC_perturbation,0.805279
3,VIP_Score,LRC_perturbation,0.785602
2,VIP_Score,Permutation,0.783680
13,Permutation,LRC_covariance,0.756078
14,LRC_perturbation,LRC_covariance,0.710877


In [106]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')